# 00 — Data cleaning

## Goal
Load the raw ImmoScout24.ch dataset and produce a clean version to build models on in the next notebooks.

No modeling or feature engineering here, just cleaning the data so later notebooks can build a trustworthy model on top of it.

## Dataset
This dataset contains ~1000 real estate listings for rent all around Switzerland. The data was scraped from https://www.immoscout24.ch/it. More info can be found [here](https://www.kaggle.com/datasets/fredeys/immoscout24-ch-switzerland-rental-property-dataset).

## Target
The target variable is `rentGross` (gross monthly rent), chosen over `rentNet` 
because it's closer to what a tenant actually pays each month. 

----------

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/ImmoScout24_CH_Rental_Property_Dataset.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 51 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        1000 non-null   int64  
 1   type                      1000 non-null   str    
 2   address                   884 non-null    str    
 3   city                      1000 non-null   str    
 4   postcode                  1000 non-null   int64  
 5   lat                       1000 non-null   float64
 6   lon                       1000 non-null   float64
 7   rentNet                   834 non-null    float64
 8   rentGross                 989 non-null    float64
 9   currency                  994 non-null    str    
 10  livingSpace               818 non-null    float64
 11  arePetsAllowed            544 non-null    object 
 12  hasFlatSharingCommunity   158 non-null    object 
 13  isUnderRoof               147 non-null    object 
 14  CHF/m2              

In [3]:
df.head()

,ID,type,address,city,postcode,lat,lon,rentNet,rentGross,currency,...,distanceKindergarten,distancePrimarySchool,distanceMotorway,distancePublicTransport,balcony,builtInKitchen,garden,offerType,personId,region
0,4000906965,"['HOUSE', 'SINGLE_HOUSE']",Alterstrasse 4,Filzbach,8757,47.120462,9.131932,1920.0,1980.0,CHF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RENT,2268787,schweiz
1,4000997183,"['APARTMENT', 'STUDIO']",Hauptstr. 30 (Nord),Döttingen,5312,47.570972,8.256332,750.0,915.0,CHF,...,700.0,700.0,NaN,170.0,NaN,NaN,NaN,RENT,679,schweiz
2,4001099145,"['APARTMENT', 'FLAT']",Falmenstrasse 2d,Uster,8610,47.352412,8.718372,1970.0,2270.0,CHF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RENT,679,schweiz
3,4001028409,"['APARTMENT', 'DUPLEX']",Am Mattenhof 2b,Kriens,6010,47.028212,8.301092,2150.0,2490.0,CHF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RENT,679,schweiz
4,4000888461,['APARTMENT'],Dorfstrasse 31,Benzenschwil,5636,47.247752,8.365612,2015.0,2300.0,CHF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RENT,1743533,schweiz


## 1. Filter by rent

This model will only consider properties for rent, so the first step is filtering 
the `offerType` field, keeping only the rows with value `"RENT"`.

In [4]:
df_rent = df[df['offerType'] == 'RENT']
df_rent.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 51 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        1000 non-null   int64  
 1   type                      1000 non-null   str    
 2   address                   884 non-null    str    
 3   city                      1000 non-null   str    
 4   postcode                  1000 non-null   int64  
 5   lat                       1000 non-null   float64
 6   lon                       1000 non-null   float64
 7   rentNet                   834 non-null    float64
 8   rentGross                 989 non-null    float64
 9   currency                  994 non-null    str    
 10  livingSpace               818 non-null    float64
 11  arePetsAllowed            544 non-null    object 
 12  hasFlatSharingCommunity   158 non-null    object 
 13  isUnderRoof               147 non-null    object 
 14  CHF/m2              

Turns out every listing in this dataset is already a rental, the filter removes 
0 rows. Kept anyway as a safety check and to make the assumption explicit.

## 2. Select the relevant columns

Keep: `type`, `postcode`, `lat`, `lon`, `livingSpace`, `numberOfRooms`, `floor`, `hasGarage`, `hasParking`, `hasBalcony`, `hasElevator` as features, `rentGross` as target

The rest of the columns are dropped either because they are listing metadata, 
or because they duplicate information.

`rentNet` is excluded because it's nearly identical to `rentGross` and would leak the target.
`CHF/m2` because it's directly derived from `rentGross / livingSpace`.
`city` has too many distinct values for a dataset of this size.
`yearBuilt` and `distance*` columns are too sparse to impute reliably.
`hasNiceView` and `isChildFriendly` are dropepd because they are subjective attributes.

In [5]:
df_clean = df_rent[['type', 'postcode', 'lat', 'lon', 'livingSpace', 'numberOfRooms', 
                    'floor', 'hasGarage', 'hasParking', 'hasBalcony', 'hasElevator', 'rentGross']].copy()
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   type           1000 non-null   str    
 1   postcode       1000 non-null   int64  
 2   lat            1000 non-null   float64
 3   lon            1000 non-null   float64
 4   livingSpace    818 non-null    float64
 5   numberOfRooms  993 non-null    float64
 6   floor          905 non-null    float64
 7   hasGarage      617 non-null    object 
 8   hasParking     582 non-null    object 
 9   hasBalcony     755 non-null    object 
 10  hasElevator    661 non-null    object 
 11  rentGross      989 non-null    float64
dtypes: float64(6), int64(1), object(4), str(1)
memory usage: 93.9+ KB


Several columns have missing values. We will handle these case by case.

## 3. Handle missing values

For the `has*` columns, we treat `Null` values as `unknown`, since we genuinely don't have that piece of information.

We filter on `livingSpace`, `numberOfRooms`, `floor` and `rentGross` to remove rows with null values in those fields.
For `livingSpace` and `rentGross` this is the only real option since the target can't be imputed, and `livingSpace` is central to the baseline model, so any imputed value would defeat the purpose.
For `numberOfRooms` the missing rate is negligible either way.
For `floor` (~100 values missing), dropping instead of imputing is a deliberate simplification for this project. 
We would rather keep this first dataset free of invented values, even at the cost of some rows. 
This can be revisited later if `floor` turns out to matter more than expected.

In [6]:
# Fill has* Null values with 'unknown'
df_clean['hasGarage'] = df_clean['hasGarage'].fillna('unknown')
df_clean['hasParking'] = df_clean['hasParking'].fillna('unknown')
df_clean['hasBalcony'] = df_clean['hasBalcony'].fillna('unknown')
df_clean['hasElevator'] = df_clean['hasElevator'].fillna('unknown')

In [7]:
# Drop Null values
df_clean = df_clean.dropna(subset=['livingSpace', 'numberOfRooms', 'floor', 'rentGross'])

In [8]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 733 entries, 1 to 999
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   type           733 non-null    str    
 1   postcode       733 non-null    int64  
 2   lat            733 non-null    float64
 3   lon            733 non-null    float64
 4   livingSpace    733 non-null    float64
 5   numberOfRooms  733 non-null    float64
 6   floor          733 non-null    float64
 7   hasGarage      733 non-null    object 
 8   hasParking     733 non-null    object 
 9   hasBalcony     733 non-null    object 
 10  hasElevator    733 non-null    object 
 11  rentGross      733 non-null    float64
dtypes: float64(6), int64(1), object(4), str(1)
memory usage: 74.4+ KB


This leaves 733 out of the original 1000 listings.

In [9]:
df_clean['hasGarage'].value_counts()
df_clean['hasGarage'].apply(type).value_counts()

hasGarage
<class 'bool'>    412
<class 'str'>     321
Name: count, dtype: int64

## 4. Fix data types

The `postcode` column is currently an integer. We convert it to a string so the model doesn't treat it as a numeric quantity with a meaningful order.
It's a categorical code, not a number to compare or scale.

The `has*` columns now mix two types within the same column: booleans for known values, and the string `"unknown"` for missing ones. 
We convert all values in these columns to strings for consistency.

In [10]:
df_clean['postcode'] = df_clean['postcode'].astype(str)

In [11]:
# Set has* data type as String
for col in ['hasGarage', 'hasParking', 'hasBalcony', 'hasElevator']:
    df_clean[col] = df_clean[col].astype(str)

In [12]:
df_clean.info()
df_clean.head()

<class 'pandas.DataFrame'>
Index: 733 entries, 1 to 999
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   type           733 non-null    str    
 1   postcode       733 non-null    str    
 2   lat            733 non-null    float64
 3   lon            733 non-null    float64
 4   livingSpace    733 non-null    float64
 5   numberOfRooms  733 non-null    float64
 6   floor          733 non-null    float64
 7   hasGarage      733 non-null    str    
 8   hasParking     733 non-null    str    
 9   hasBalcony     733 non-null    str    
 10  hasElevator    733 non-null    str    
 11  rentGross      733 non-null    float64
dtypes: float64(6), str(6)
memory usage: 74.4 KB


,type,postcode,lat,lon,livingSpace,numberOfRooms,floor,hasGarage,hasParking,hasBalcony,hasElevator,rentGross
1,"['APARTMENT', 'STUDIO']",5312,47.570972,8.256332,45.0,1.0,-1.0,True,unknown,unknown,True,915.0
2,"['APARTMENT', 'FLAT']",8610,47.352412,8.718372,88.0,3.5,2.0,True,True,unknown,unknown,2270.0
3,"['APARTMENT', 'DUPLEX']",6010,47.028212,8.301092,117.0,3.5,4.0,unknown,unknown,unknown,unknown,2490.0
5,"['APARTMENT', 'FLAT']",8623,47.329232,8.820222,86.0,4.5,0.0,True,unknown,unknown,unknown,2133.0
7,"['APARTMENT', 'LOFT']",8877,47.112022,9.217332,155.0,2.5,5.0,True,True,True,unknown,2500.0


## 5. Save the cleaned dataset

Output used as the starting point for `01_baseline_regression.ipynb`.

In [14]:
df_clean.to_csv('../data/ImmoScout24_Clean.csv', index=False)